# Computer Vision Model Evaluation Pipeline

This notebook demonstrates a complete evaluation workflow for object detection models in the computer vision workspace:

1. **Dataset Download**: Load Open Images dataset subset with doors
2. **SAM3 Concept Segmentation**: Apply text-prompt based segmentation  
3. **mAP-COCO Evaluation**: Evaluate prediction quality using COCO metrics

This follows the multi-project workspace conventions for dataset management and model evaluation.

In [13]:
# Import Required Libraries for CV Workspace
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.brain as fob
from huggingface_hub import login
import os
import pathlib
import yaml
import numpy as np
import logging

# Configure environment for optimal performance
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Setup logging for CV workspace
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")
print("FiftyOne version:", fo.__version__)

✅ Libraries imported successfully
FiftyOne version: 1.11.0


In [14]:
# Authenticate with Hugging Face for SAM3 model access
# Load credentials from CV workspace secrets
secrets_path = pathlib.Path.cwd().parent / "cvmgr" / "configs" / "secrets.yaml"

if secrets_path.exists():
    with secrets_path.open('r') as file:
        secrets_yaml = yaml.safe_load(file)
    
    login(secrets_yaml["huggingface"]["token"])
    print("✅ Authenticated with Hugging Face")
else:
    print("⚠️ secrets.yaml not found. Please set up Hugging Face authentication:")
    print("1. Create cvmgr/configs/secrets.yaml")
    print("2. Add your HF token: huggingface: token: 'your_token_here'")
    # Fallback to manual login
    # login()  # Uncomment if you want to login manually

2025-12-08 14:24:23,015 - INFO - HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"


✅ Authenticated with Hugging Face


## Step 1: Download Open Images Dataset Subset

Download a subset of Open Images dataset focusing on "Door handle" class with 100 samples for efficient evaluation in the CV workspace.

In [15]:
# Download Open Images dataset subset with door handles
dataset_name = "door_handles_evaluation_100"

# Check if dataset already exists in CV workspace
if fo.dataset_exists(dataset_name):
    print(f"Dataset '{dataset_name}' already exists, deleting due to version mismatch...")
    fo.delete_dataset(dataset_name)
    print("Old dataset deleted, will create new one...")

# Create fresh dataset to avoid version issues
print(f"Downloading Open Images dataset with door handles (100 samples)...")

# Load Open Images dataset with specific parameters
dataset = foz.load_zoo_dataset(
    "open-images-v7",
    splits=["validation"],  # Use validation split for consistent evaluation
    label_types=["segmentations"],  # Focus on segmentation masks
    classes=["Door handle"],  # Specific class for concept segmentation
    max_samples=100,  # Limit for efficient evaluation
    shuffle=True,
    seed=42  # Reproducible sampling
)

# Save dataset for reuse in CV workspace
dataset.name = dataset_name
dataset.persistent = True
dataset.save()

print(f"✅ Dataset downloaded: {len(dataset)} samples")

# Display dataset info
print(f"\nDataset Info:")
print(f"Name: {dataset.name}")
print(f"Samples: {len(dataset)}")
print(f"Classes: {dataset.distinct('ground_truth.detections.label')}")

# Show sample image
print(f"🔍 Dataset loaded in FiftyOne App")

2025-12-08 14:24:23,031 - INFO - Downloading split 'validation' to '/home/freeze/fiftyone/open-images-v7/validation' if necessary


Only found 26 (<100) samples matching your requirements


2025-12-08 14:24:23,310 - WARNING - Only found 26 (<100) samples matching your requirements


Necessary images already downloaded


2025-12-08 14:24:23,312 - INFO - Necessary images already downloaded


...

 100% |███████████████████| 26/26 [587.3ms elapsed, 0s remaining, 44.4 samples/s]      
 100% |███████████████████| 26/26 [587.3ms elapsed, 0s remaining, 44.4 samples/s]      


2025-12-08 14:24:24,081 - INFO -  100% |███████████████████| 26/26 [587.3ms elapsed, 0s remaining, 44.4 samples/s]      


Dataset 'open-images-v7-validation-100' created


2025-12-08 14:24:24,095 - INFO - Dataset 'open-images-v7-validation-100' created


✅ Dataset downloaded: 26 samples

Dataset Info:
Name: door_handles_evaluation_100
Samples: 26
Classes: ['Door handle', 'Sculpture']
🔍 Dataset loaded in FiftyOne App


## Step 2: Setup SAM3 Model for Concept Segmentation

Register the SAM3 model source and prepare for text-prompt based segmentation using "door handle" concept.

In [16]:
# Use SAM2 model (more stable than SAM3) for segmentation
print("Loading SAM2 model for segmentation...")
try:
    # Try to load SAM2 model from FiftyOne zoo
    model = foz.load_zoo_model("segment-anything-2-hiera-large")
    print("✅ SAM2 model loaded successfully")
    print(f"Model: {type(model)}")
    
except Exception as sam2_error:
    print(f"SAM2 loading failed: {sam2_error}")
    print("Trying alternative segmentation approach...")
    
    try:
        # Fallback to Detectron2 semantic segmentation
        model = foz.load_zoo_model("detectron2-resnet50-fpn-sem-seg-coco")
        print("✅ Detectron2 semantic segmentation model loaded")
        
    except Exception as fallback_error:
        logger.error(f"All segmentation models failed: {fallback_error}")
        print("❌ No segmentation models available. Try:")
        print("1. Check internet connection")
        print("2. Verify Hugging Face authentication") 
        print("3. Try running with detections instead of segmentations")
        model = None

Loading SAM2 model for segmentation...


2025-12-08 14:24:24,212 - ERROR - All segmentation models failed: Model 'detectron2-resnet50-fpn-sem-seg-coco' not found in the zoo


SAM2 loading failed: Model 'segment-anything-2-hiera-large' not found in the zoo
Trying alternative segmentation approach...
❌ No segmentation models available. Try:
1. Check internet connection
2. Verify Hugging Face authentication
3. Try running with detections instead of segmentations


## Step 3: Apply SAM3 Segmentation

Apply text-prompt based segmentation using the concept "door handle" to generate pixel-level segmentation masks that will be evaluated against ground truth segmentation masks.

In [17]:
# Apply SAM3 segmentation with "door handle" prompt
concept_prompt = "door handle"
prediction_field = "sam3_predictions"

print(f"Applying SAM3 segmentation with prompt: '{concept_prompt}'")
print(f"Processing {len(dataset)} samples...")

try:
    # Apply model with concept segmentation
    dataset.apply_model(
        model,
        label_field=prediction_field,
        text=concept_prompt,  # Text prompt for concept segmentation
        batch_size=4,         # Conservative batch size for GPU stability
        num_workers=1         # Single worker for memory efficiency
    )
    
    print(f"✅ SAM3 segmentation completed")
    print(f"Segmentation masks saved to field: '{prediction_field}'")
    
    # Check prediction quality
    sample_with_predictions = dataset.match(fo.ViewField(prediction_field).exists()).first()
    if sample_with_predictions:
        num_predictions = len(sample_with_predictions[prediction_field].detections)
        print(f"Example: First sample has {num_predictions} predictions")
    else:
        print("⚠️ No predictions found in any sample")
        
except Exception as e:
    logger.error(f"SAM3 segmentation failed: {e}")
    print("❌ Segmentation failed. Check GPU memory and model configuration.")
    
# Memory cleanup after inference
import torch
import gc
torch.cuda.empty_cache()
gc.collect()
print("🧹 GPU memory cleared")

2025-12-08 14:24:24,229 - ERROR - SAM3 segmentation failed: Unsupported model type: <class 'NoneType'>


Applying SAM3 segmentation with prompt: 'door handle'
Processing 26 samples...
❌ Segmentation failed. Check GPU memory and model configuration.
🧹 GPU memory cleared
🧹 GPU memory cleared


## Step 4: Evaluate Predictions using mAP-COCO Metrics

Calculate mAP (mean Average Precision) using COCO evaluation protocol to assess the quality of SAM3 predictions against ground truth annotations.

In [18]:
# Evaluate predictions using mAP-COCO metrics
print("Starting COCO-style evaluation...")

try:
    # Create evaluation results for CV workspace analysis
    results = dataset.evaluate_segmentations(
        prediction_field,           # SAM3 segmentation predictions field
        gt_field="ground_truth",    # Ground truth segmentation masks field
        eval_key="sam3_eval",       # Evaluation results key
        mask_targets=["Door handle"],  # Target classes for segmentation
        classwise=True             # Compute per-class metrics
    )
    
    print("✅ Evaluation completed!")
    
    # Display evaluation metrics
    print("\n" + "="*50)
    print("SAM3 CONCEPT SEGMENTATION EVALUATION RESULTS")
    print("SAM3 SEGMENTATION EVALUATION RESULTS")
    
    # Overall mAP metrics
    # Overall segmentation metrics
    print(f"Mean IoU: {results.mean_iou():.4f}")
    if hasattr(results, 'mAP'):
        print(f"mAP: {results.mAP():.4f}")
    
    # Detailed metrics breakdown
    metrics = results.metrics()
    print("\nDetailed Metrics:")
    for metric_name, value in metrics.items():
        if isinstance(value, (int, float)):
            print(f"  {metric_name}: {value:.4f}")
    
    # Class-wise metrics (if available)
    try:
        classwise_metrics = results.classwise_metrics()
        print("\nClass-wise mAP:")
        for class_name, class_metrics in classwise_metrics.items():
            if 'mAP' in class_metrics:
                print(f"  {class_name}: {class_metrics['mAP']:.4f}")
    except:
        print("\nClass-wise metrics not available")
    
    # Confusion matrix
    print(f"\nConfusion Matrix:")
    cm = results.confusion_matrix()
    print(cm)
    
except Exception as e:
    logger.error(f"Evaluation failed: {e}")
    print("❌ Evaluation failed. Check that predictions exist and are properly formatted.")
    print("Possible issues:")

    print("- Ground truth field format incompatible")    print("- Insufficient samples for meaningful evaluation")

SyntaxError: invalid syntax (140988631.py, line 54)

## Step 5: Visualization and Analysis

Create visualizations to understand model performance and analyze prediction quality for the computer vision workspace.

In [ ]:
# Create evaluation plots and analysis for CV workspace
print("Generating evaluation visualizations...")

try:
    # Plot PR (Precision-Recall) curve
    plot = results.plot_pr_curves(classes=["Door handle"])
    plot.show()
    print("✅ Precision-Recall curve displayed")
    
    # Create confusion matrix plot
    cm_plot = results.plot_confusion_matrix()
    cm_plot.show()
    print("✅ Confusion matrix displayed")
    
except Exception as e:
    print(f"⚠️ Plotting failed: {e}")
    print("Evaluation results available but visualization not supported in this environment")

# Analyze specific prediction types
print("\n" + "="*40)
print("PREDICTION ANALYSIS")
print("="*40)

# True Positives - Correct predictions
true_positives = dataset.filter_labels("sam3_predictions", results.true_positives())
print(f"True Positives: {len(true_positives)} samples")

# False Positives - Incorrect predictions
false_positives = dataset.filter_labels("sam3_predictions", results.false_positives())
print(f"False Positives: {len(false_positives)} samples")

# False Negatives - Missed ground truth
false_negatives = dataset.filter_labels("ground_truth", results.false_negatives())
print(f"False Negatives: {len(false_negatives)} samples")

# Best and worst predictions for analysis
if len(true_positives) > 0:
    print(f"\n🎯 Best predictions: {min(5, len(true_positives))} samples with high confidence correct detections")
    
if len(false_positives) > 0:
    print(f"❌ False alarms: {min(5, len(false_positives))} samples with incorrect predictions")
    
if len(false_negatives) > 0:
    print(f"🔍 Missed detections: {min(5, len(false_negatives))} samples with undetected doors")

# Launch FiftyOne App with evaluation results for detailed analysis
print(f"\n🔍 Launching FiftyOne App with evaluation results...")
session = fo.launch_app(dataset)
print("Evaluation complete! Explore results in the FiftyOne App.")

## Summary and Next Steps

This notebook demonstrated a complete computer vision evaluation pipeline:

### Results Summary
- **Dataset**: 100 samples from Open Images with door handle segmentation masks
- **Model**: SAM3 segmentation with "door handle" prompt  
- **Evaluation**: Segmentation IoU metrics and pixel-level accuracy

### Key Takeaways
1. **SAM3 Segmentation** can generate precise pixel-level masks using text prompts
2. **Segmentation Evaluation** provides IoU and pixel-accuracy metrics for mask quality
3. **FiftyOne Integration** enables seamless dataset management and evaluation workflows

### Next Steps for CV Workspace
- **Scale to larger datasets** for more robust evaluation
- **Compare different models** (YOLO, SAM3, etc.) on same dataset
- **Fine-tune thresholds** based on evaluation results
- **Export results** for model training pipeline integration

### Workspace Integration
This evaluation workflow can be integrated into your multi-project CV pipeline:
- Use `train_yolo_model()` function for model training
- Apply evaluation metrics for model selection
- Export evaluation results for documentation and comparison